# Tumour Slide Comparison

**Temporary notebook** — loops through every tumour slide and displays:
1. Tissue outline with tumour annotation overlay
2. Patch grid zoomed into the tumour region

Use this to pick the clearest examples for the blog.

Each slide is downloaded, visualised, then deleted before the next one is fetched — so peak RAM stays low even across all 50+ slides.

In [ ]:
# === Setup: clone repo and install dependencies ===

from pathlib import Path

REPO_URL = 'https://github.com/balintstewart77/camelyon16-pathology.git'
REPO_DIR = Path('/content/camelyon16-pathology')

if not REPO_DIR.exists():
    !git clone {REPO_URL} /content/camelyon16-pathology
else:
    print('Repository already present.')

%cd /content/camelyon16-pathology

!apt-get install -y openslide-tools > /dev/null 2>&1
!pip install -q -r requirements.txt

print('Environment ready.')

In [ ]:
# === Mount Google Drive ===

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === Imports ===

import gc
import os
import traceback

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import openslide
from PIL import Image

from config import DEFAULT_CONFIG
from src.data import list_s3_files, download_file_from_s3, cleanup_file
from src.data.tissue_mask import compute_foreground_mask
from src.data.tumor_polygons import load_tumor_polygons, classify_patch
from src.data.patch_extraction import sample_grid_coordinates
from src.visualisation import (
    visualise_tissue_outline,
    visualise_patches_grid,
    find_zoom_region_by_coords,
)

print('All imports OK.')

In [ ]:
# === List tumour slides ===

all_slides = list_s3_files(DEFAULT_CONFIG.data.s3_images, '.tif')
tumor_slides = sorted([f for f in all_slides if 'tumor' in f.lower()])

all_annotations = list_s3_files(DEFAULT_CONFIG.data.s3_annotations, '.xml')
annotation_names = {f for f in all_annotations}

print(f'Found {len(tumor_slides)} tumour slides.')
print()

# Only keep slides that have a matching annotation file
slides_with_annotations = []
slides_missing_annotations = []
for slide_name in tumor_slides:
    stem = Path(slide_name).stem          # e.g. 'tumor_005'
    xml_name = f'{stem}.xml'
    if xml_name in annotation_names:
        slides_with_annotations.append((slide_name, xml_name))
    else:
        slides_missing_annotations.append(slide_name)

print(f'Slides with annotations: {len(slides_with_annotations)}')
if slides_missing_annotations:
    print(f'Slides WITHOUT annotations (skipped): {slides_missing_annotations}')

In [ ]:
# === Configuration ===

# Set to None to process every slide, or a list of names to process a subset.
# Example: SLIDES_TO_PROCESS = ['tumor_005.tif', 'tumor_010.tif']
SLIDES_TO_PROCESS = None   # None = all slides with annotations

PATCH_SIZE    = 224
GRID_STRIDE   = 224        # stride for the regular grid
TMP_DIR       = '/tmp'     # slides are downloaded here then deleted

CLASS_COLOURS = {1: 'green', 2: 'orange', 3: 'red'}
CLASS_LABELS  = {1: 'Normal', 2: 'Boundary', 3: 'Pure Tumour'}

if SLIDES_TO_PROCESS is not None:
    slides_with_annotations = [
        (s, x) for s, x in slides_with_annotations if s in SLIDES_TO_PROCESS
    ]

print(f'Will process {len(slides_with_annotations)} slides.')

In [ ]:
# === Main loop ===
#
# For each tumour slide:
#   1. Download slide + annotation
#   2. Compute tissue mask
#   3. Load tumour polygons
#   4. Display: tissue outline with tumour overlay
#   5. Classify grid patches and display zoom
#   6. Delete downloaded files and free memory
#
# Failures are caught and printed so one bad slide won't stop the loop.

failed_slides = []

for idx, (slide_name, xml_name) in enumerate(slides_with_annotations):
    slide_id = Path(slide_name).stem
    print(f'\n{"-" * 60}')
    print(f'[{idx + 1}/{len(slides_with_annotations)}]  {slide_id}')
    print(f'{"-" * 60}')

    slide_path = None
    xml_path   = None
    slide      = None

    try:
        # ── 1. Download ────────────────────────────────────────────────
        print('  Downloading slide...', end=' ', flush=True)
        slide_path = download_file_from_s3(
            DEFAULT_CONFIG.data.s3_images, slide_name, TMP_DIR
        )
        print('done.')

        print('  Downloading annotation...', end=' ', flush=True)
        xml_path = download_file_from_s3(
            DEFAULT_CONFIG.data.s3_annotations, xml_name, TMP_DIR
        )
        print('done.')

        # ── 2. Open slide and compute tissue mask ──────────────────────
        slide = openslide.OpenSlide(slide_path)
        dims  = slide.dimensions
        print(f'  Dimensions: {dims[0]:,} × {dims[1]:,}  |  '
              f'Levels: {slide.level_count}')

        mask = compute_foreground_mask(slide)
        tissue_pct = mask.sum() / mask.size
        print(f'  Tissue coverage: {tissue_pct:.1%}')

        # ── 3. Load tumour polygons ────────────────────────────────────
        polygons = load_tumor_polygons(xml_path)
        total_tumor_area = sum(p.area for p in polygons)
        slide_area = dims[0] * dims[1]
        print(f'  Tumour polygons: {len(polygons)}  |  '
              f'Tumour area: {total_tumor_area / slide_area:.2%} of slide')

        # ── 4. Tissue outline + tumour overlay ─────────────────────────
        visualise_tissue_outline(
            slide, mask,
            tumor_polygons=polygons,
            title=f'{slide_id} — Tissue outline & tumour annotations',
            figsize=(10, 10),
        )
        plt.show()
        plt.close('all')

        # ── 5. Grid patch classification ───────────────────────────────
        coords = sample_grid_coordinates(
            slide, mask, patch_size=PATCH_SIZE, stride=GRID_STRIDE
        )

        coords_by_class = {1: [], 2: [], 3: []}
        for x, y in coords:
            label = classify_patch(x, y, polygons, patch_size=PATCH_SIZE)
            coords_by_class[label].append((x, y))

        print(f'  Grid patches: {len(coords):,} total')
        for cls, cls_coords in sorted(coords_by_class.items()):
            print(f'    Class {cls} ({CLASS_LABELS[cls]}): {len(cls_coords):,}')

        # ── 6. Zoomed patch grid ───────────────────────────────────────
        zoom_coords = (
            coords_by_class.get(3, [])
            or coords_by_class.get(2, [])
            or coords_by_class.get(1, [])
        )

        if zoom_coords:
            zoom_region = find_zoom_region_by_coords(zoom_coords, region_size=10000)
            visualise_patches_grid(
                slide,
                coords_by_class,
                zoom_region=zoom_region,
                patch_size=PATCH_SIZE,
                class_colours=CLASS_COLOURS,
                class_labels=CLASS_LABELS,
                title=f'{slide_id} — Patch grid (zoomed to tumour)',
                linewidth=1.5,
                figsize=(14, 12),
            )
            plt.show()
            plt.close('all')
        else:
            print('  No patch coordinates found — skipping grid visualisation.')

    except Exception as exc:
        print(f'\n  ERROR processing {slide_id}:')
        traceback.print_exc()
        failed_slides.append((slide_id, str(exc)))

    finally:
        # ── 7. Clean up ────────────────────────────────────────────────
        # Close the slide handle before deleting the file
        if slide is not None:
            try:
                slide.close()
            except Exception:
                pass
            slide = None

        for path in (slide_path, xml_path):
            if path is not None:
                try:
                    cleanup_file(path)
                except Exception:
                    pass

        # Release memory
        del slide, slide_path, xml_path
        try:
            del mask, polygons, coords, coords_by_class, zoom_coords, zoom_region
        except NameError:
            pass
        gc.collect()
        plt.close('all')

# ── Summary ──────────────────────────────────────────────────────────────────
print(f'\n{"=" * 60}')
print(f'Done. Processed {len(slides_with_annotations)} slides.')
if failed_slides:
    print(f'\nFailed slides ({len(failed_slides)}):')
    for name, err in failed_slides:
        print(f'  {name}: {err}')
else:
    print('All slides processed successfully.')